# Train the Card Model (run in Google Colab)

Trains a YOLO26n model (newest Ultralytics YOLO, built for edge devices) to read card **rank** (A, 2-10, J, Q, K) and face-down cards (`DOWN`) from the robot's camera, then exports it to ONNX for the Jetson.

**Before you start:** *Runtime > Change runtime type > T4 GPU*.

Steps:
1. Install, upload our `card_dataset` (from `card_data_collector.ipynb`)
2. Train on our own pictures for ~1 hour with heavy augmentation, starting from pretrained YOLO26n weights (saved to Google Drive so a disconnect doesn't lose it)
3. Check accuracy (goal from the proposal: 90%)
4. Export `best.onnx` and download it

In [ ]:
!pip install -q -U ultralytics

import os, glob, random, shutil, zipfile
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

CLASSES = ['A', '2', '3', '4', '5', '6', '7', '8', '9', '10', 'J', 'Q', 'K', 'DOWN']   # same order as the collector, DOWN = face-down card
IMGSZ = 416   # small enough for the Jetson Nano
START_WEIGHTS = 'yolo26n.pt'   # newest YOLO, nano size. Use 'yolo11n.pt' if this fails to download
os.chdir('/content')

## 1. Upload our dataset
On the Jetson, zip the folder first (in a Jupyter terminal, inside `HackRPI26/`):
```
zip -r card_dataset.zip card_dataset
```
Download that zip to your laptop, then upload it with the cell below.

In [ ]:
from google.colab import files
uploaded = files.upload()   # pick card_dataset.zip

with zipfile.ZipFile('card_dataset.zip') as z:
    z.extractall('/content')
print("Images:", len(glob.glob('/content/card_dataset/images/*.jpg')))

## Split ours into train / val (80 / 20)
Val pictures are never trained on, so the accuracy number at the end is honest.

In [ ]:
def split_dataset(src, dst, val_frac=0.2, seed=0):
    names = sorted(os.path.splitext(os.path.basename(p))[0] for p in glob.glob(f'{src}/images/*.jpg'))
    random.Random(seed).shuffle(names)
    n_val = max(1, int(len(names) * val_frac))
    for split, group in [('val', names[:n_val]), ('train', names[n_val:])]:
        os.makedirs(f'{dst}/{split}/images', exist_ok=True)
        os.makedirs(f'{dst}/{split}/labels', exist_ok=True)
        for n in group:
            shutil.copy(f'{src}/images/{n}.jpg', f'{dst}/{split}/images/')
            shutil.copy(f'{src}/labels/{n}.txt', f'{dst}/{split}/labels/')
    return len(names) - n_val, n_val


def write_yaml(path, root):
    with open(path, 'w') as f:
        f.write(f"path: {root}\ntrain: train/images\nval: val/images\n")
        f.write(f"nc: {len(CLASSES)}\nnames: {CLASSES}\n")


shutil.rmtree('/content/ours', ignore_errors=True)
n_train, n_val = split_dataset('/content/card_dataset', '/content/ours')
write_yaml('/content/ours.yaml', '/content/ours')
print("train:", n_train, " val:", n_val)

## 2. Save runs to Google Drive
Colab can disconnect during a long run. Saving to Drive keeps `best.pt` safe and lets you resume.
Keep the Colab tab open while it trains (free Colab stops idle sessions).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RUN_DIR = '/content/drive/MyDrive/blackjack_runs'
os.makedirs(RUN_DIR, exist_ok=True)
print("Runs will be saved to", RUN_DIR)

## 3. Train (~1 hour, heavy augmentation)
Same model (YOLO26n), same image size (416, what the Jetson runs), **only our training pictures**. The val pictures are never trained on.

Every epoch the training pictures get randomly changed, so the model sees "new" versions each time:

| Trick | Setting | Why it helps us |
|---|---|---|
| Time budget | `time=0.9` h | Uses the hour; the learning-rate schedule is fit to the time |
| Brightness / color | `hsv_v=0.5, hsv_s=0.6, hsv_h=0.015` | Lab lighting changes, shadows from the arm |
| Rotation | `degrees=15` | Cards aren't dealt perfectly straight |
| Shift / zoom | `translate=0.2, scale=0.5` | Cards anywhere in view, different sizes per pose |
| Shear / perspective | `shear=5, perspective=0.0005` | Camera tilt differs between poses |
| Mosaic | `mosaic=1.0`, off for the last 20 epochs | Glues 4 pictures together: more cards per picture, teaches multi-card scenes |
| MixUp / CutMix | `mixup=0.15, cutmix=0.15` | Blends/patches pictures: partly covered cards, less overfitting |
| No flips | `fliplr=0, flipud=0` | A mirrored card isn't a real card |
| Cosine LR + warmup | `cos_lr=True, warmup_epochs=5` | Smoother training over a long run |
| Weight decay | `weight_decay=0.001` | Less overfitting on a small dataset |
| No early stop | `patience=0` | Run the full hour; `best.pt` still keeps the best epoch |
| Cache in RAM | `cache='ram'` | Faster epochs = more epochs in the hour |

**Estimate:** with ~250-400 pictures, a T4 does roughly 4-8 s per epoch, so ~400-800 epochs in the hour.
Heavy augmentation is what makes that many epochs useful instead of just memorizing.

If the session dies, run the **Resume** cell below instead of starting over.

In [ ]:
TRAIN_HOURS = 0.9   # leaves a few minutes for checking + export

model = YOLO(START_WEIGHTS)
model.train(
    data='/content/ours.yaml', imgsz=IMGSZ,
    time=TRAIN_HOURS, epochs=1000, patience=0,   # time wins: epochs get fit to the hour
    batch=32, cache='ram', workers=4, seed=0,
    cos_lr=True, warmup_epochs=5, weight_decay=0.001,
    # lighting
    hsv_h=0.015, hsv_s=0.6, hsv_v=0.5,
    # geometry
    degrees=15, translate=0.2, scale=0.5, shear=5, perspective=0.0005,
    fliplr=0.0, flipud=0.0,
    # mixing pictures
    mosaic=1.0, close_mosaic=20, mixup=0.15, cutmix=0.15,
    project=RUN_DIR, name='long_aug', exist_ok=True)
BEST = f'{RUN_DIR}/long_aug/weights/best.pt'

### Resume (only if Colab disconnected mid-training)
Rerun the setup, upload, split and Drive cells first, then set `RESUME = True` and run this one. Leave it `False` normally (safe with Run all).

In [ ]:
RESUME = False   # set True only if Colab disconnected during training

if RESUME:
    BEST = f'{RUN_DIR}/long_aug/weights/best.pt'
    YOLO(f'{RUN_DIR}/long_aug/weights/last.pt').train(resume=True)

## 4. Check accuracy
Card reading accuracy on the val pictures = the model's most confident box has the right rank.
Proposal goal: **at least 90%**.

In [ ]:
model = YOLO(BEST)
model.val(data='/content/ours.yaml', imgsz=IMGSZ)

right, total, wrong = 0, 0, []
for img in sorted(glob.glob('/content/ours/val/images/*.jpg')):
    true = CLASSES[int(open(img.replace('/images/', '/labels/').replace('.jpg', '.txt')).read().split()[0])]
    r = model(img, imgsz=IMGSZ, verbose=False)[0]
    pred = CLASSES[int(r.boxes.cls[r.boxes.conf.argmax()])] if len(r.boxes) else 'none'
    total += 1
    if pred == true:
        right += 1
    else:
        wrong.append((img, true, pred))

print(f"Card reading accuracy: {right}/{total} = {100*right/max(total,1):.1f}%")
for img, true, pred in wrong:
    print(f"  {os.path.basename(img)}: true {true}, predicted {pred}")

## See some predictions

In [ ]:
imgs = random.sample(glob.glob('/content/ours/val/images/*.jpg'), min(6, n_val))
plt.figure(figsize=(15, 8))
for i, img in enumerate(imgs):
    r = model(img, imgsz=IMGSZ, verbose=False)[0]
    plt.subplot(2, 3, i + 1)
    plt.imshow(r.plot()[:, :, ::-1])
    plt.axis('off')
plt.show()

## 5. Export for the Jetson
ONNX with opset 12 so the Jetson's older software can load it. Downloads `best.pt` and `best.onnx`; put them in `HackRPI26/models/` on the Jetson.

In [ ]:
onnx_path = YOLO(BEST).export(format='onnx', imgsz=IMGSZ, opset=12, simplify=True)
print("Exported:", onnx_path)
shutil.copy(onnx_path, RUN_DIR)   # backup copy in Drive
files.download(BEST)
files.download(onnx_path)